# 10. Terminations (when should the team stop?)

A team takes turns — but **when should it stop**?
Without a stop rule, agents could talk **forever** (and cost money).

A **termination condition** is the rule that tells the team to **stop**.

## Real-life analogy

A **meeting** needs an ending rule, like:

- "Stop when we all **agree**." (someone says APPROVE)
- "Stop after **10 minutes**." (a limit, no matter what)

Without an ending rule, the meeting never ends. Teams are the same.

## Common termination conditions

| Condition | Stops when... |
|-----------|---------------|
| `TextMentionTermination("APPROVE")` | A message contains the word "APPROVE" |
| `MaxMessageTermination(6)` | 6 messages have been sent (a safety limit) |
| Combine with `|` | **Either** rule is met |

Combining is smart: stop when approved **OR** after a max number of messages (so it never runs forever).

In [1]:
from dotenv import load_dotenv
load_dotenv()

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

writer = AssistantAgent(name="writer", model_client=model_client,
                        system_message="Write a short thank-you note. Improve it if asked.")
reviewer = AssistantAgent(name="reviewer", model_client=model_client,
                          system_message="If the note is polite, reply APPROVE. Else suggest one change.")

# Stop when APPROVE appears OR after 6 messages (whichever comes first)
stop = TextMentionTermination("APPROVE") | MaxMessageTermination(6)

team = RoundRobinGroupChat([writer, reviewer], termination_condition=stop)

result = await team.run(task="Write a thank-you note to a teacher.")

print("Total messages:", len(result.messages))
print("Why it stopped:", result.stop_reason)

Total messages: 3
Why it stopped: Text 'APPROVE' mentioned


## Key points to remember

- A **termination condition** tells a team **when to stop**.
- `TextMentionTermination("APPROVE")` stops when a **keyword** appears.
- `MaxMessageTermination(n)` stops after **n messages** (a safety limit).
- Combine rules with **`|`** to stop when **either** happens.
- Always set a stop rule so a team **never runs forever**.
- Check **`result.stop_reason`** to see why it stopped.